# Problem 2: 注意力机制和 Transformer 块 (Attention and Transformer Blocks)

在本问题中，你将实现 Transformer 模型的核心组件：注意力机制和 Transformer 块。

**主要参考论文**: [Vaswani et al., 2017](https://arxiv.org/abs/1706.03762) - "Attention Is All You Need"

## 2.1 缩放点积注意力 (Scaled Dot-Product Attention)

**目标**: 实现 `run_scaled_dot_product_attention` 函数。

给定查询 (Query, Q)、键 (Key, K)、值 (Value, V) 矩阵，缩放点积注意力定义为：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

其中：
- $Q \in \mathbb{R}^{n \times d_k}$ 是查询矩阵
- $K \in \mathbb{R}^{m \times d_k}$ 是键矩阵
- $V \in \mathbb{R}^{m \times d_v}$ 是值矩阵
- $d_k$ 是键/查询的维度
- $n$ 是查询的数量
- $m$ 是键/值的数量

**当提供 mask 时**：mask 中的位置应该被设为 $-\infty$（在 softmax 之前），
这样这些位置的注意力权重就会变为 0。

**实现要求**:
- 不能使用 `torch.nn.MultiheadAttention` 或 `F.scaled_dot_product_attention`
- 需要支持批量输入（多个前导维度）
- Mask 是可选的，当为 None 时不应用 mask
- Mask 中 True 的位置表示应该被 mask 掉

**对应函数**: `tests/adapters.py` 中的 `run_scaled_dot_product_attention(Q, K, V, mask)`

## 2.2 旋转位置编码 (Rotary Positional Embeddings, RoPE)

**目标**: 实现 `run_rope` 函数。

**参考论文**: [Su et al., 2021](https://arxiv.org/abs/2104.09864) - "RoFormer: Enhanced Transformer with Rotary Position Embedding"

RoPE 通过旋转操作将位置信息注入到查询和键中。

### 核心思想

对于维度 $d_k$，我们定义频率：

$$\theta_i = \Theta^{-2i/d_k}, \quad i \in [0, d_k/2)$$

其中 $\Theta$ 是一个超参数（通常为 10000）。

对于位置 $m$，旋转角度为：

$$\phi(m, i) = m \cdot \theta_i$$

对于查询或键向量 $x \in \mathbb{R}^{d_k}$，我们将它分成两部分：
- $x_{even}$：偶数索引位置的元素 $[x_0, x_2, x_4, ...]$
- $x_{odd}$：奇数索引位置的元素 $[x_1, x_3, x_5, ...]$

旋转操作定义为：

$$\text{RoPE}(x, m) = \begin{bmatrix} x_0 \\ x_2 \\ x_4 \\ \vdots \\ x_1 \\ x_3 \\ x_5 \\ \vdots \end{bmatrix} \odot \begin{bmatrix} \cos(\phi(m, 0)) \\ \cos(\phi(m, 1)) \\ \cos(\phi(m, 2)) \\ \vdots \\ \sin(\phi(m, 0)) \\ \sin(\phi(m, 1)) \\ \sin(\phi(m, 2)) \\ \vdots \end{bmatrix} + \begin{bmatrix} -x_1 \\ -x_3 \\ -x_5 \\ \vdots \\ x_0 \\ x_2 \\ x_4 \\ \vdots \end{bmatrix} \odot \begin{bmatrix} \sin(\phi(m, 0)) \\ \sin(\phi(m, 1)) \\ \sin(\phi(m, 2)) \\ \vdots \\ \cos(\phi(m, 0)) \\ \cos(\phi(m, 1)) \\ \cos(\phi(m, 2)) \\ \vdots \end{bmatrix}$$

**实现步骤**:
1. 预计算所有位置的旋转角度 $\phi(m, i)$（可选优化）
2. 将输入张量重塑为 [..., seq_len, d_k//2, 2] 以分离奇偶索引
3. 应用旋转公式
4. 重新 reshape 回原始形状

**实现要求**:
- 需要支持批量输入和任意数量的前导维度
- 需要支持任意位置（不一定是连续的 0, 1, 2, ...）
- `token_positions` 参数指定每个 token 的位置

**对应函数**: `tests/adapters.py` 中的 `run_rope(d_k, theta, max_seq_len, in_query_or_key, token_positions)`

## 2.3 多头自注意力 (Multi-Head Self-Attention)

**目标**: 实现 `run_multihead_self_attention` 函数。

多头注意力允许模型同时关注不同的表示子空间。
对于 $h$ 个注意力头，我们定义：

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

其中每个头计算为：

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**维度说明**:
- 输入 $X \in \mathbb{R}^{batch \times seq\_len \times d_{model}}$
- 每个头的维度 $d_k = d_v = d_{model} / h$
- $W_i^Q, W_i^K \in \mathbb{R}^{d_k \times d_{model}}$
- $W_i^V \in \mathbb{R}^{d_v \times d_{model}}$
- $W^O \in \mathbb{R}^{d_{model} \times d_{model}}$

**批量实现优化**:

不要为每个头单独计算，而是将所有头的投影合并为一次矩阵乘法：

- 将所有 $W_i^Q$ 垂直堆叠成 $W^Q \in \mathbb{R}^{(h \cdot d_k) \times d_{model}}$
- 类似地处理 $W^K$ 和 $W^V$
- 一次性计算所有头的 Q, K, V
- Reshape 为 $[batch, seq\_len, h, d_k]$ 并转置为 $[batch, h, seq\_len, d_k]$
- 应用缩放点积注意力
- 合并头并通过 $W^O$ 投影

**实现要求**:
- **不使用 RoPE**（这是不带位置编码的版本）
- 使用优化的批量实现
- 不使用 `torch.nn.MultiheadAttention`

**对应函数**: `tests/adapters.py` 中的 `run_multihead_self_attention(d_model, num_heads, q_proj_weight, k_proj_weight, v_proj_weight, o_proj_weight, in_features)`

## 2.4 带 RoPE 的多头自注意力 (Multi-Head Self-Attention with RoPE)

**目标**: 实现 `run_multihead_self_attention_with_rope` 函数。

这与 2.3 的多头注意力相同，但在计算注意力之前对 Q 和 K 应用 RoPE。

**实现步骤**:
1. 使用投影权重计算 Q, K, V
2. Reshape 为多头格式 $[batch, h, seq\_len, d_k]$
3. 对 Q 和 K 应用 RoPE
4. 计算缩放点积注意力
5. 合并头并通过输出投影

**关键点**:
- RoPE 应该在每个头的 $d_k$ 维度上应用
- $d_k = d_{model} / num\_heads$ 必须是偶数（因为 RoPE 需要成对处理）
- `token_positions` 参数指定每个 token 的位置索引

**对应函数**: `tests/adapters.py` 中的 `run_multihead_self_attention_with_rope(d_model, num_heads, max_seq_len, theta, q_proj_weight, k_proj_weight, v_proj_weight, o_proj_weight, in_features, token_positions)`

## 2.5 Transformer 块 (Transformer Block)

**目标**: 实现 `run_transformer_block` 函数。

Transformer 块是 Transformer 的核心构建模块，包含自注意力和前馈网络。

**Pre-Norm 结构**（我们使用的是 pre-norm）：

$$\text{output} = X + \text{MHA}(\text{LN}_1(X))$$
$$\text{output} = \text{output} + \text{FFN}(\text{LN}_2(\text{output}))$$

**详细步骤**:
1. **第一个 RMSNorm**: $\text{norm}_1 = \text{RMSNorm}(X)$
2. **多头自注意力**: $\text{attn\_out} = \text{MHA}(\text{norm}_1)$ (使用 RoPE)
3. **第一个残差连接**: $\text{hidden}_1 = X + \text{attn\_out}$
4. **第二个 RMSNorm**: $\text{norm}_2 = \text{RMSNorm}(\text{hidden}_1)$
5. **前馈网络**: $\text{ffn\_out} = \text{SwiGLU}(\text{norm}_2)$
6. **第二个残差连接**: $\text{output} = \text{hidden}_1 + \text{ffn\_out}$

**权重字典结构**:
```python
weights = {
    'attn.q_proj.weight': ...,   # (d_model, d_model)
    'attn.k_proj.weight': ...,   # (d_model, d_model)
    'attn.v_proj.weight': ...,   # (d_model, d_model)
    'attn.output_proj.weight': ..., # (d_model, d_model)
    'ln1.weight': ...,           # (d_model,)
    'ffn.w1.weight': ...,        # (d_model, d_ff)
    'ffn.w2.weight': ...,        # (d_ff, d_model)
    'ffn.w3.weight': ...,        # (d_model, d_ff)
    'ln2.weight': ...,           # (d_model,)
}
```

**实现要求**:
- 使用 pre-norm 结构（先归一化，再应用子层）
- 使用带 RoPE 的多头自注意力
- 使用 SwiGLU 前馈网络
- 使用 RMSNorm（不是 LayerNorm）

**对应函数**: `tests/adapters.py` 中的 `run_transformer_block(d_model, num_heads, d_ff, max_seq_len, theta, weights, in_features)`

## 2.6 Transformer 语言模型 (Transformer Language Model)

**目标**: 实现 `run_transformer_lm` 函数。

Transformer 语言模型将所有组件组合在一起，预测下一个 token。

**整体架构**:

1. **Token Embeddings**: 将 token ID 映射为嵌入向量
   $$E = \text{Embedding}(\text{token\_ids}) \in \mathbb{R}^{batch \times seq\_len \times d_{model}}$$

2. **Transformer Layers**: 堆叠多个 Transformer 块
   $$H = \text{TransformerBlock}_1(E)$$
   $$H = \text{TransformerBlock}_2(H)$$
   $$...$$
   $$H = \text{TransformerBlock}_{num\_layers}(H)$$

3. **Final RMSNorm**: 应用最终的层归一化
   $$H = \text{RMSNorm}(H)$$

4. **LM Head**: 投影到词汇表
   $$\text{logits} = H \cdot W_{lm\_head}^T \in \mathbb{R}^{batch \times seq\_len \times vocab\_size}$$

**权重字典结构**:
```python
weights = {
    'token_embeddings.weight': ...,  # (vocab_size, d_model)
    'layers.0.attn.q_proj.weight': ...,  # 第 0 层
    'layers.0.attn.k_proj.weight': ..., 
    'layers.0.attn.v_proj.weight': ..., 
    'layers.0.attn.output_proj.weight': ..., 
    'layers.0.ln1.weight': ..., 
    'layers.0.ffn.w1.weight': ..., 
    'layers.0.ffn.w2.weight': ..., 
    'layers.0.ffn.w3.weight': ..., 
    'layers.0.ln2.weight': ..., 
    # ... 更多层 ...
    'layers.{num_layers-1}....': ...,  # 最后一层
    'ln_final.weight': ...,  # (d_model,)
    'lm_head.weight': ...,  # (vocab_size, d_model)
}
```

**权重共享**: 通常 `lm_head.weight` 和 `token_embeddings.weight` 是相同的（权重共享）。

**输入/输出**:
- 输入: `in_indices` $\in \mathbb{Z}^{batch \times seq\_len}$ (token IDs)
- 输出: `logits` $\in \mathbb{R}^{batch \times seq\_len \times vocab\_size}$ (每个位置的下一个 token 的未归一化 logit)

**实现要求**:
- 堆叠 `num_layers` 个 Transformer 块
- 每个块都使用 RoPE
- 最终输出是未归一化的 logits（不需要应用 softmax）

**对应函数**: `tests/adapters.py` 中的 `run_transformer_lm(vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta, weights, in_indices)`